# using bge-large-en-v1.5 to get dense Embeding of 

In [ ]:
!pip install -q sentence-transformers

In [1]:
import json
import numpy as np
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

# ============================================================
# Paths
# ============================================================

SEMANTIC_MATCHES_PATH = (
    "/kaggle/input/datasets/d202511054/"
    "clef-2026-checkthat-task-2-data/my data/semantic_matches.json"
)

NEW_DATA_PATH = (
    "/kaggle/input/datasets/d202511054/"
    "clef-2026-checkthat-task-2-data/my data/new_data_merged.json"
)

OUTPUT_NPZ = "/kaggle/working/new_data_dense_embeddings.npz"
OUTPUT_META = "/kaggle/working/new_data_dense_embeddings_metadata.json"

# ============================================================
# Load model
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

model = SentenceTransformer(
    "BAAI/bge-large-en-v1.5",
    device=device
)

# ============================================================
# Load semantic matches
# ============================================================

with open(SEMANTIC_MATCHES_PATH, "r", encoding="utf-8") as f:
    semantic_matches = json.load(f)

new_ids = {
    item["new_id"]
    for item in semantic_matches
}

print("Unique new_ids:", len(new_ids))

# ============================================================
# Load merged data
# ============================================================

with open(NEW_DATA_PATH, "r", encoding="utf-8") as f:
    merged_data = json.load(f)

print("Merged samples:", len(merged_data))

id_lookup = {
    sample["id"]: sample
    for sample in merged_data
}

matched_claims = [
    id_lookup[nid]
    for nid in new_ids
    if nid in id_lookup
]

print("Matched claims:", len(matched_claims))

assert len(matched_claims) == 6469

# ============================================================
# Verify trace counts
# ============================================================

trace_distribution = {}

for sample in matched_claims:
    n = len(sample["Reasoning_traces"])
    trace_distribution[n] = trace_distribution.get(n, 0) + 1

print("Trace distribution:")
print(trace_distribution)

# ============================================================
# Generate embeddings
# ============================================================

embeddings_dict = {}
metadata = []

for sample in tqdm(matched_claims):

    claim_id = sample["id"]

    claim_text = sample["claim"]

    traces = sample["Reasoning_traces"]

    texts = [claim_text] + traces

    emb = model.encode(
        texts,
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    emb = emb.astype(np.float32)

    embeddings_dict[f"id_{claim_id}"] = emb

    metadata.append({
        "id": claim_id,
        "num_traces": len(traces),
        "num_embeddings": len(texts)
    })

# ============================================================
# Save NPZ
# ============================================================

np.savez_compressed(
    OUTPUT_NPZ,
    **embeddings_dict
)

# ============================================================
# Save metadata
# ============================================================

with open(OUTPUT_META, "w", encoding="utf-8") as f:
    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nSaved:")
print(OUTPUT_NPZ)
print(OUTPUT_META)

print("\nExample shape:")

sample_key = list(embeddings_dict.keys())[0]

print(
    sample_key,
    embeddings_dict[sample_key].shape
)

Device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Unique new_ids: 6469
Merged samples: 10558
Matched claims: 6469
Trace distribution:
{20: 6210, 15: 259}


  0%|          | 0/6469 [00:00<?, ?it/s]


Saved:
/kaggle/working/new_data_dense_embeddings.npz
/kaggle/working/new_data_dense_embeddings_metadata.json

Example shape:
id_1 (21, 1024)


# get dense embedding of Test Dataset

In [ ]:
import json
import numpy as np
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

# ============================================================
# Paths
# ============================================================

TEST_DATA_PATH = (
    "/kaggle/input/datasets/d202511054/"
    "clef-2026-checkthat-task-2-data/test.json"
)

OUTPUT_NPZ = "/kaggle/working/test_dense_embeddings.npz"
OUTPUT_META = "/kaggle/working/test_dense_embeddings_metadata.json"

# ============================================================
# Load model
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

model = SentenceTransformer(
    "BAAI/bge-large-en-v1.5",
    device=device
)

# ============================================================
# Load test dataset
# ============================================================

with open(TEST_DATA_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Test samples:", len(test_data))

# ============================================================
# Verify reasoning trace counts
# ============================================================

trace_distribution = {}

for sample in test_data:
    n = len(sample["Reasoning_traces"])
    trace_distribution[n] = trace_distribution.get(n, 0) + 1

print("\nTrace distribution:")
print(trace_distribution)

# ============================================================
# Generate embeddings
# ============================================================

embeddings_dict = {}
metadata = []

for sample in tqdm(test_data):

    query_id = sample["query_id"]

    claim_text = sample["claim"]

    traces = sample["Reasoning_traces"]

    # claim first, then all reasoning traces
    texts = [claim_text] + traces

    emb = model.encode(
        texts,
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    emb = emb.astype(np.float32)

    # Same structure as training embeddings
    embeddings_dict[f"id_{query_id}"] = emb

    metadata.append({
        "query_id": query_id,
        "num_traces": len(traces),
        "num_embeddings": len(texts)
    })

# ============================================================
# Save NPZ
# ============================================================

np.savez_compressed(
    OUTPUT_NPZ,
    **embeddings_dict
)

# ============================================================
# Save metadata
# ============================================================

with open(OUTPUT_META, "w", encoding="utf-8") as f:
    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

# ============================================================
# Verify saved structure
# ============================================================

print("\nSaved files:")
print(OUTPUT_NPZ)
print(OUTPUT_META)

sample_key = list(embeddings_dict.keys())[0]

print("\nExample:")
print("Key:", sample_key)
print("Shape:", embeddings_dict[sample_key].shape)

print("\nExpected shape:")
print("(1 claim + 15 traces, 1024 dimensions)")

0


# getting dense embeddings of Validatiation Data

In [ ]:
import json
import numpy as np
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

# ============================================================
# Paths
# ============================================================

VALIDATION_DATA_PATH = (
    "/kaggle/input/datasets/d202511054/"
    "clef-2026-checkthat-task-2-data/validation.json"
)

OUTPUT_NPZ = "/kaggle/working/validation_dense_embeddings.npz"
OUTPUT_META = "/kaggle/working/validation_dense_embeddings_metadata.json"

# ============================================================
# Load model
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

model = SentenceTransformer(
    "BAAI/bge-large-en-v1.5",
    device=device
)

# ============================================================
# Load validation dataset
# ============================================================

with open(VALIDATION_DATA_PATH, "r", encoding="utf-8") as f:
    validation_data = json.load(f)

print("Validation samples:", len(validation_data))

# ============================================================
# Verify trace counts
# ============================================================

trace_distribution = {}

for sample in validation_data:
    n = len(sample["Reasoning_traces"])
    trace_distribution[n] = trace_distribution.get(n, 0) + 1

print("\nTrace distribution:")
print(trace_distribution)

# ============================================================
# Generate embeddings
# ============================================================

embeddings_dict = {}
metadata = []

for idx, sample in enumerate(tqdm(validation_data)):

    sample_id = idx

    claim_text = sample["claim"]

    traces = sample["Reasoning_traces"]

    # claim first, then traces
    texts = [claim_text] + traces

    emb = model.encode(
        texts,
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    emb = emb.astype(np.float32)

    embeddings_dict[f"id_{sample_id}"] = emb

    metadata.append({
        "sample_id": sample_id,
        "num_traces": len(traces),
        "num_embeddings": len(texts)
    })

# ============================================================
# Save NPZ
# ============================================================

np.savez_compressed(
    OUTPUT_NPZ,
    **embeddings_dict
)

# ============================================================
# Save metadata
# ============================================================

with open(OUTPUT_META, "w", encoding="utf-8") as f:
    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

# ============================================================
# Verify saved structure
# ============================================================

print("\nSaved files:")
print(OUTPUT_NPZ)
print(OUTPUT_META)

sample_key = list(embeddings_dict.keys())[0]

print("\nExample:")
print("Key:", sample_key)
print("Shape:", embeddings_dict[sample_key].shape)

print("\nExpected shape:")

embedding_dim = embeddings_dict[sample_key].shape[1]

print(
    f"(1 claim + reasoning traces, {embedding_dim} dimensions)"
)